# 三位置 GBM 整合分析 — GSE241037 + GSE274987

| 数据集 | 样本数 | 区域 | 平台 |
|---|---:|---|---|
| GSE241037 | 12 | Peripheral / Edge / Core | 10X |
| GSE274987 | 37 | Peripheral / Edge / Core | ParseBio + Multiome |

流程: 读取 → 统一元数据 → QC → Doublet → 合并 → HVG → PCA → Harmony → Leiden → UMAP → 保存

说明: GSE274987 优先读取现成 h5ad，并在需要时修复旧格式的 `uns/log1p/base`。

In [ ]:
import gc
import shutil
import warnings
from pathlib import Path

import anndata as ad
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1
sc.set_figure_params(dpi=100, color_map="viridis_r", transparent=False, frameon=False)
%config InlineBackend.figure_format = "retina"

print(f"scanpy {sc.__version__}")
print(f"anndata {ad.__version__}")

In [ ]:
min_cells = 3
min_umi = 1000
min_genes = 200
max_genes = 8000
mt_cap = 10
hvg_n_top = 2000
pca_n_comps = 50
batch_key = "sample"
hvg_batch_key = "sample"  # sample 太碎，HVG 按 dataset 做
leiden_res_list = [0.5, 0.8, 1.0]
leiden_default = 0.5

BASE_DIR = Path(".").resolve()
DATA241_DIR = BASE_DIR / "GSE241037"
DATA274_DIR = BASE_DIR / "GSE274987"
WRITE_DIR = BASE_DIR / "write" / "0_integrate_two_datasets"
FIG_DIR = BASE_DIR / "figures" / "0_integrate_two_datasets"
WRITE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(FIG_DIR)

print(BASE_DIR)
print(WRITE_DIR)
print(FIG_DIR)

## 1. 读取 GSE241037

In [ ]:
sample_sheet_241 = pd.read_csv(DATA241_DIR / "GSE241037_sample_anno.csv", index_col=0)
sample_sheet_241["dataset"] = "GSE241037"
sample_sheet_241["Region"] = sample_sheet_241["tissue"].map({"Peripheral": "PT", "Edge": "TE", "Core": "TC"})
sample_sheet_241["sample"] = "GSE241037_" + sample_sheet_241["geo_accession"].astype(str)
sample_sheet_241

In [ ]:
import gzip
from scipy.io import mmread

def read_selected_barcodes(barcode_file, keep_idx):
    keep_idx = np.asarray(keep_idx, dtype=int)
    barcodes = []
    ptr = 0
    with gzip.open(barcode_file, "rt") as handle:
        for i, line in enumerate(handle):
            if ptr >= len(keep_idx):
                break
            if i == keep_idx[ptr]:
                barcodes.append(line.strip())
                ptr += 1
    return barcodes

def load_raw_matrix_sample(matrix_file, feature_file, barcode_file):
    matrix = mmread(gzip.open(matrix_file)).tocsr()
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    n_genes = np.asarray((matrix > 0).sum(axis=0)).ravel()
    keep = (counts >= min_umi) & (n_genes >= min_genes)
    keep_idx = np.flatnonzero(keep)
    barcodes = read_selected_barcodes(barcode_file, keep_idx)

    features = pd.read_csv(feature_file, sep="\t", header=None)
    var = pd.DataFrame(index=features[1].astype(str))
    var.index.name = None
    var["gene_ids"] = features[0].astype(str).values
    if features.shape[1] > 2:
        var["feature_types"] = features[2].astype(str).values

    adata = ad.AnnData(X=matrix[:, keep].T.tocsr(), var=var)
    adata.obs_names = barcodes
    adata.var_names_make_unique()
    print(f"raw 预筛: {matrix.shape[1]} -> {adata.n_obs}")
    return adata

def load_10x_prefix_sample(data_dir, prefix):
    matrix_file = data_dir / f"{prefix}matrix.mtx.gz"
    feature_file = data_dir / f"{prefix}features.tsv.gz"
    barcode_file = data_dir / f"{prefix}barcodes.tsv.gz"

    with gzip.open(matrix_file, "rt") as handle:
        next(handle)
        next(handle)
        n_genes, n_barcodes, _ = map(int, next(handle).strip().split())

    if n_barcodes > 100000:
        adata = load_raw_matrix_sample(matrix_file, feature_file, barcode_file)
    else:
        adata = sc.read_10x_mtx(data_dir, var_names="gene_symbols", make_unique=True, prefix=prefix, cache=False)

    if not sparse.issparse(adata.X):
        adata.X = sparse.csr_matrix(adata.X)
    adata.X = adata.X.astype(np.int32)
    return adata

adatas_241 = []
for _, row in sample_sheet_241.iterrows():
    gsm = str(row["geo_accession"])
    matrix_file = next(iter(DATA241_DIR.glob(f"{gsm}_*_matrix.mtx.gz")), None)
    if matrix_file is None:
        print(f"找不到 {gsm} 的 matrix 文件")
        continue

    prefix = matrix_file.name.replace("matrix.mtx.gz", "")
    adata = load_10x_prefix_sample(DATA241_DIR, prefix)
    adata.obs["barcode"] = adata.obs_names.astype(str)
    adata.obs_names = [f"GSE241037_{gsm}_{bc}" for bc in adata.obs_names]
    adata.obs["dataset"] = "GSE241037"
    adata.obs["dataset_source"] = "GSE241037"
    adata.obs["Patient"] = str(row["patient"])
    adata.obs["patient_id"] = "GSE241037_" + str(row["patient"])
    adata.obs["Sample"] = gsm
    adata.obs["sample_raw"] = gsm
    adata.obs["sample"] = "GSE241037_" + gsm
    adata.obs["sample_id"] = adata.obs["sample"]
    adata.obs["title"] = str(row["title"])
    adata.obs["Region"] = row["Region"]
    adata.obs["tissue_type"] = str(row["tissue"])
    adata.obs["Group"] = str(row["tissue"])
    adata.obs["Batch"] = "Batch_10X"
    adata.obs["Platform"] = "10X"
    adatas_241.append(adata)
    print(gsm, adata.shape)

print("GSE241037 样本数:", len(adatas_241))

## 2. 读取 GSE274987

In [ ]:
sample_sheet_274 = pd.read_csv(DATA274_DIR / "GSE274987.csv")
sample_sheet_274["Patient"] = sample_sheet_274["Patient"].astype(str)
sample_sheet_274["sample_raw"] = sample_sheet_274["Sample"].astype(str)
sample_sheet_274["sample"] = "GSE274987_" + sample_sheet_274["sample_raw"]
sample_sheet_274["patient_id"] = "GSE274987_" + sample_sheet_274["Patient"]
sample_sheet_274

In [ ]:
def load_gse274987_h5ad(src_path, repaired_path):
    try:
        return ad.read_h5ad(src_path)
    except Exception as err:
        print(f"直接读取失败: {err}")

    need_copy = (not repaired_path.exists()) or (repaired_path.stat().st_mtime < src_path.stat().st_mtime)
    if need_copy:
        shutil.copy2(src_path, repaired_path)
        with h5py.File(repaired_path, "r+") as f:
            if "uns" in f and "log1p" in f["uns"] and "base" in f["uns"]["log1p"]:
                attrs = dict(f["uns"]["log1p"]["base"].attrs)
                if attrs.get("encoding-type") == "null":
                    del f["uns"]["log1p"]["base"]
    return ad.read_h5ad(repaired_path)

repaired_274 = WRITE_DIR / "adata_GSE274987.repaired.h5ad"
adata_274 = load_gse274987_h5ad(DATA274_DIR / "adata_GSE274987.h5ad", repaired_274)
adata_274.var_names_make_unique()
adata_274.X = adata_274.layers["counts"].copy()
adata_274.obs["Patient"] = adata_274.obs["Patient"].astype(str)
adata_274.obs["sample_raw"] = adata_274.obs["Sample"].astype(str)
adata_274.obs["sample"] = "GSE274987_" + adata_274.obs["sample_raw"]
adata_274.obs["sample_id"] = adata_274.obs["sample"]
adata_274.obs["patient_id"] = "GSE274987_" + adata_274.obs["Patient"]
adata_274.obs["dataset"] = "GSE274987"
adata_274.obs["dataset_source"] = "GSE274987"
adata_274.shape

In [ ]:
sheet_map_274 = sample_sheet_274.set_index("sample_raw")[["Region", "tissue_type", "Batch", "Platform", "n_cells"]]
adata_274.obs = adata_274.obs.join(sheet_map_274, on="sample_raw", rsuffix="_sheet")
for col in ["Region", "tissue_type", "Batch", "Platform"]:
    sheet_col = f"{col}_sheet"
    if sheet_col in adata_274.obs.columns:
        adata_274.obs[col] = adata_274.obs[sheet_col].fillna(adata_274.obs[col].astype(str))
        adata_274.obs.drop(columns=sheet_col, inplace=True)
adata_274.obs.rename(columns={"n_cells": "n_cells_expected"}, inplace=True)
adata_274.obs["Group"] = adata_274.obs["tissue_type"].astype(str)

adatas_274 = []
for sample in sorted(adata_274.obs["sample"].unique()):
    adata_sub = adata_274[adata_274.obs["sample"] == sample].copy()
    adatas_274.append(adata_sub)
    print(sample, adata_sub.shape)

print("GSE274987 样本数:", len(adatas_274))

## 3. 统一元数据

In [ ]:
all_adatas = adatas_241 + adatas_274
sample_sheet_cols = [
    "dataset", "dataset_source", "Patient", "patient_id", "Sample", "sample_raw",
    "sample", "sample_id", "Region", "tissue_type", "Group", "Batch", "Platform"
]

sample_sheet = pd.concat(
    [adata.obs[sample_sheet_cols].drop_duplicates() for adata in all_adatas],
    ignore_index=True,
).sort_values(["dataset", "Patient", "Sample"])
sample_sheet.to_csv(WRITE_DIR / "sample_sheet.csv", index=False)

print("总样本数:", sample_sheet["sample"].nunique())
print(sample_sheet.groupby(["dataset", "tissue_type"]).size().unstack(fill_value=0))
sample_sheet.head()

## 4. 质量控制

In [ ]:
def annotate_qc(adata):
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.match(r"^RP[SL]\d")
    adata.var["hb"] = adata.var_names.str.match(r"^HB[^P]")
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo", "hb"],
        percent_top=None,
        log1p=False,
        inplace=True,
    )

def apply_qc(adata):
    adata = adata.copy()
    annotate_qc(adata)
    keep = (
        (adata.obs["total_counts"] >= min_umi)
        & (adata.obs["n_genes_by_counts"] >= min_genes)
        & (adata.obs["n_genes_by_counts"] <= max_genes)
        & (adata.obs["pct_counts_mt"] <= mt_cap)
    )
    return adata[keep].copy()

qc_adatas = []
qc_rows = []
for adata in all_adatas:
    n_before = adata.n_obs
    adata_qc = apply_qc(adata)
    n_after = adata_qc.n_obs
    qc_adatas.append(adata_qc)
    qc_rows.append({
        "dataset": adata.obs["dataset"].iat[0],
        "sample": adata.obs["sample"].iat[0],
        "tissue_type": adata.obs["tissue_type"].iat[0],
        "n_before": n_before,
        "n_after": n_after,
        "retention": n_after / max(n_before, 1),
    })
    print(adata.obs["sample"].iat[0], f"{n_before} -> {n_after}")

qc_stats = pd.DataFrame(qc_rows)
qc_stats.to_csv(WRITE_DIR / "qc_by_sample.csv", index=False)
qc_stats.groupby(["dataset", "tissue_type"])[["n_before", "n_after"]].sum()

## 5. Doublet 与合并

In [ ]:
singlet_adatas = []
doublet_rows = []
for adata in qc_adatas:
    n_before = adata.n_obs
    expected_rate = min(max(n_before * 0.008 / 1000, 0.02), 0.15)
    try:
        sc.pp.scrublet(adata, expected_doublet_rate=expected_rate, random_state=42)
    except Exception as err:
        print(adata.obs["sample"].iat[0], err)
        adata.obs["predicted_doublet"] = False
        adata.obs["doublet_score"] = 0.0

    singlet = adata[~adata.obs["predicted_doublet"]].copy()
    singlet_adatas.append(singlet)
    doublet_rows.append({
        "dataset": adata.obs["dataset"].iat[0],
        "sample": adata.obs["sample"].iat[0],
        "n_before": n_before,
        "n_doublets": int(adata.obs["predicted_doublet"].sum()),
        "n_after": singlet.n_obs,
    })
    print(adata.obs["sample"].iat[0], f"{n_before} -> {singlet.n_obs}")

doublet_df = pd.DataFrame(doublet_rows)
doublet_df["doublet_rate"] = doublet_df["n_doublets"] / doublet_df["n_before"].clip(lower=1)
doublet_df.to_csv(WRITE_DIR / "doublet_by_sample.csv", index=False)
doublet_df.groupby("dataset")[["n_before", "n_doublets", "n_after"]].sum()

In [ ]:
adata = sc.concat(singlet_adatas, join="inner", merge="unique")
sc.pp.filter_genes(adata, min_cells=min_cells)
print(adata)
print(adata.obs["dataset"].value_counts())
print(adata.obs["tissue_type"].value_counts())

adata.write(WRITE_DIR / "adata.h5ad")
print(WRITE_DIR / "adata.h5ad")

## 6. 归一化与特征选择

In [ ]:
hvg_n_top = 3000
hvg_batch_key = "dataset"
adata = sc.read_h5ad(WRITE_DIR / "adata.h5ad")
adata.var_names_make_unique()
adata.layers["counts"] = adata.X.copy()

print("HVG override")
print("hvg_n_top:", hvg_n_top)
print("hvg_batch_key:", hvg_batch_key)
print(adata)

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=hvg_n_top,
    flavor="seurat_v3",
    batch_key=hvg_batch_key,
    layer="counts",
)
print("HVG batch:", hvg_batch_key)
print("HVG 数:", int(adata.var["highly_variable"].sum()))

sc.pl.highly_variable_genes(adata, show=False)
plt.savefig(FIG_DIR / "hvg_selection.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 7. PCA Harmony UMAP Leiden

In [ ]:
import harmonypy as hm

adata_hvg = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=pca_n_comps, random_state=42)
adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"].copy()
adata.uns["pca"] = adata_hvg.uns["pca"]

ho = hm.run_harmony(adata.obsm["X_pca"], adata.obs, batch_key, random_state=42)
adata.obsm["X_pca_harmony"] = ho.Z_corr.T
sc.pp.neighbors(adata, use_rep="X_pca_harmony", random_state=42)
sc.tl.umap(adata, random_state=42)
for res in leiden_res_list:
    key = f"leiden_res_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key, random_state=42, flavor="igraph", n_iterations=2)
adata.obs["leiden"] = adata.obs[f"leiden_res_{leiden_default}"]
del adata_hvg, ho
gc.collect()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
sc.pl.umap(adata, color="leiden", ax=axes[0, 0], show=False, legend_loc="on data", title="Leiden")
sc.pl.umap(adata, color="dataset", ax=axes[0, 1], show=False, title="Dataset")
sc.pl.umap(adata, color="tissue_type", ax=axes[1, 0], show=False, title="Tissue")
sc.pl.umap(adata, color="Platform", ax=axes[1, 1], show=False, title="Platform")
plt.tight_layout()
plt.savefig(FIG_DIR / "umap_integrated_overview.pdf", dpi=300, bbox_inches="tight")
plt.show()

adata.write(WRITE_DIR / "adata2.h5ad")
print(adata)
print(WRITE_DIR / "adata2.h5ad")

## 8. 细胞注释

In [ ]:
marker_gene_sets = {
    'Oligodendrocyte': ['MAG', 'CLDN11', 'APLP1', 'TMEM144', 'CNDP1', 'EDIL3', 'LARP6', 'MOG', 'AMER2', 'TUBB4A'],
    'MES-like': ['VIM', 'CD44', 'CHI3L1', 'HILPDA', 'DDIT3', 'ENO2'],
    'AC-like': ['GFAP', 'S100B', 'HOPX', 'SLC1A3', 'MLC1'],
    'OPC-like': ['PLP1', 'OLIG1', 'OMG', 'TNR', 'ALCAM'],
    'NPC-like': ['SOX4', 'SOX11', 'DCX', 'CD24', 'STMN1', 'STMN2'],
    'NK/T': ['CD3D', 'CD3E', 'TRBC2', 'CD3G', 'CD2', 'TRAC', 'IL7R', 'TRBC1', 'LCK', 'SKAP1', 'CD48'],
    'Prol.NK/T': ['MKI67', 'TOP2A', 'CD3D', 'CD3E', 'TRBC2', 'CD3G'],
    'Pericyte': ['COL1A2', 'COL6A2', 'PDGFRB', 'COL3A1', 'EDNRA', 'LUM', 'COL1A1', 'PLAC9', 'FRZB'],
    'Endothelial': ['ESAM', 'FLT1', 'RAMP2', 'VWF', 'EGFL7', 'SLC9A3R2', 'CAVIN2', 'ADGRL4', 'ABLIM1', 'ABCB1'],
    'Monocyte': ['FCN1', 'VCAN', 'S100A8', 'S100A9'],
    'Microglia': ['TMEM119', 'CX3CR1', 'TREM2', 'GPNMB', 'CTSD', 'CD68'],
    'Macrophage': ['SPP1', 'APOE', 'GPNMB', 'CTSD', 'CD68', 'C1QA', 'C1QB'],
    'CDC': ['FCER1A', 'HLA-DPB1', 'CD83', 'AREG'],
    'Neutrophil': ['S100A8', 'FCGR3B', 'IFITM2'],
}

MIN_SCORE_DIFF = 0.05

In [ ]:
marker_gene_sets_use = {}
marker_rows = []
gene_to_ct = {}

for ct, genes in marker_gene_sets.items():
    present = [g for g in genes if g in adata.var_names]
    if present:
        marker_gene_sets_use[ct] = present
    marker_rows.append({
        'celltype': ct,
        'n_marker': len(genes),
        'n_present': len(present),
        'missing': ', '.join([g for g in genes if g not in adata.var_names]),
    })
    for gene in present:
        gene_to_ct.setdefault(gene, []).append(ct)

marker_df = pd.DataFrame(marker_rows)
overlap_genes = {g: v for g, v in gene_to_ct.items() if len(v) > 1}

marker_df[['celltype', 'n_marker', 'n_present']]
print('重叠 marker:', overlap_genes)

In [ ]:
for ct, genes in marker_gene_sets_use.items():
    if genes:
        sc.tl.score_genes(adata, genes, score_name=f'score_{ct}')

score_cols = [f'score_{ct}' for ct in marker_gene_sets_use if marker_gene_sets_use[ct]]
scores_df = adata.obs[score_cols].copy()
sorted_scores = np.sort(scores_df.to_numpy(), axis=1)
top1 = sorted_scores[:, -1]
top2 = sorted_scores[:, -2]
score_diff = top1 - top2

score_winner = scores_df.idxmax(axis=1).str.replace('score_', '', regex=False)
score_winner = pd.Series(score_winner.to_numpy(), index=adata.obs_names, dtype='object')
score_winner.loc[score_diff < MIN_SCORE_DIFF] = 'Ambiguous'

cluster_anno = {}
cluster_records = []
for cl in adata.obs['leiden'].cat.categories:
    mask = adata.obs['leiden'] == cl
    votes = score_winner.loc[mask]
    non_ambig = votes[votes != 'Ambiguous']
    if len(non_ambig) == 0:
        label = 'Unknown'
        support = 0
    else:
        top_votes = non_ambig.value_counts()
        label = top_votes.index[0]
        support = top_votes.iloc[0] / len(votes)
    cluster_anno[cl] = label
    cluster_records.append({
        'leiden': cl,
        'celltype': label,
        'support_fraction': support,
        'n_cells': int(mask.sum()),
        'n_ambiguous': int((votes == 'Ambiguous').sum()),
    })

cluster_anno_df = pd.DataFrame(cluster_records)
adata.obs['celltype'] = pd.Categorical(adata.obs['leiden'].map(cluster_anno))
cluster_anno_df.to_csv(WRITE_DIR / 'cluster_celltype_annotation.csv', index=False)
# 正式粗注释只写入 celltype；score_winner 只在本 cell 内用于投票。
print(adata.obs['celltype'].value_counts())
display(cluster_anno_df)


In [ ]:
sc.pl.dotplot(
    adata,
    marker_gene_sets_use,
    groupby='celltype',
    standard_scale='var',
    show=False,
    dendrogram=False,
    figsize=(18, 6),
)
plt.savefig(FIG_DIR / 'marker_dotplot_celltype.pdf', dpi=300, bbox_inches='tight')
plt.show()

marker_list = []
for genes in marker_gene_sets_use.values():
    for gene in genes:
        if gene not in marker_list:
            marker_list.append(gene)

sc.pl.dotplot(
    adata,
    marker_list,
    groupby='leiden',
    standard_scale='var',
    show=False,
    figsize=(18, 6),
)
plt.savefig(FIG_DIR / 'marker_dotplot_leiden.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color='celltype', ax=axes[0], show=False, title='Celltype')
sc.pl.umap(adata, color='tissue_type', ax=axes[1], show=False, title='Tissue')
plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_celltype_annotation.pdf', dpi=300, bbox_inches='tight')
plt.show()

# 清理旧版派生注释列，只保留正式 celltype
drop_cols = ['predicted_celltype', 'annotation_confidence', 'celltype_merged', 'myeloid_celltype', 'celltype_refined']
for col in drop_cols:
    if col in adata.obs.columns:
        del adata.obs[col]

adata.write(WRITE_DIR / 'adata2.h5ad')
print(WRITE_DIR / 'cluster_celltype_annotation.csv')
print(WRITE_DIR / 'adata2.h5ad')